# Project: Advanced Credit Risk Prediction
## Notebook 03: Feature Engineering and Preprocessing

[![Author](https://img.shields.io/badge/Author-Prakash%20Ukhalkar-blue.svg)](https://github.com/prakash-ukhalkar) [![GitHub Repository](https://img.shields.io/badge/GitHub-Repo-lightgrey)](https://github.com/prakash-ukhalkar/Advanced-Credit-Risk-Prediction) [![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![Scikit-Learn](https://img.shields.io/badge/Scikit--Learn-Latest-orange)](https://scikit-learn.org/) [![Imbalanced-Learn](https://img.shields.io/badge/Imbalanced--Learn-Latest-yellow)](https://imbalanced-learn.org/)

---

**Objective:** Develop a robust, reproducible data preprocessing pipeline tailored to the Dual-Track datasets.

**Introduction:** This stage focuses on transforming raw statistical insights into model-ready features. We implement a rigorous pipeline involving multivariate imputation for corporate financial data, robust scaling to handle outliers, and synthetic minority over-sampling (SMOTE) to rectify class imbalances. Crucially, we enforce a splitting strategy prior to any transformation to ensure zero data leakage from the test set into our training pipeline.

## Step 1: Library Imports
We leverage `scikit-learn` for industrial-strength preprocessing and `imblearn` for handling the severe class imbalances localized in our corporate track. The use of `IterativeImputer` (MICE) allows us to preserve the statistical relationships between financial ratios during the reconstruction of missing values.

In [4]:
import joblib
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

## Step 2: Data Re-Ingestion (Dual-Track)
To maintain notebook independence, we reload representative samples from the Individual (Retail) and Financial (Corporate) tracks. Note that we target specifically identified CSVs from our previous profiling stage.

In [5]:
RAW_DATA_DIR = os.path.join('..', 'data', 'raw')

# Representative Retail and Corporate Data
retail_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'Credit Risk_Individual_Kaggel_26022026.csv'))
corp_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'Corporate Default Financial Dataset.csv'))

print(f"Loaded Retail Shape: {retail_df.shape}")
print(f"Loaded Corporate Shape: {corp_df.shape}")

Loaded Retail Shape: (1000, 21)
Loaded Corporate Shape: (92048, 24)


## Step 3: Leakage-Safe Train-Test Split
**Methodological Requirement:** Before applying imputation, scaling, or encoding, we must partition the data. Transformations must only 'learn' parameters (mean, variance, categories) from the training set. Applying these on the full dataset causes **Data Leakage**, where info from the test set 'bleeds' into training, resulting in artificially high but fragile performance metrics.

In [6]:
# Dynamically identify targets
target_retail = next((col for col in retail_df.columns if 'default' in col.lower() or 'target' in col.lower() or 'risk' in col.lower()), retail_df.columns[-1])
target_corp = next((col for col in corp_df.columns if 'default' in col.lower() or 'target' in col.lower() or 'risk' in col.lower()), corp_df.columns[-1])

# Retail Split
X_ret = retail_df.drop(columns=[target_retail])
y_ret = retail_df[target_retail]
# Professional Label Encoding: Map 'bad' -> 1 (Risk), 'good' -> 0 (Safe)
y_ret = y_ret.map({'bad': 1, 'good': 0}).astype(int)
X_ret_train, X_ret_test, y_ret_train, y_ret_test = train_test_split(X_ret, y_ret, test_size=0.2, random_state=42)

# Corporate Split
X_corp = corp_df.drop(columns=[target_corp])
y_corp = corp_df[target_corp]
X_corp_train, X_corp_test, y_corp_train, y_corp_test = train_test_split(X_corp, y_corp, test_size=0.2, random_state=42)

print("Split Summary:")
print(f"Retail Train/Test: {len(X_ret_train)} / {len(X_ret_test)}")
print(f"Corporate Train/Test: {len(X_corp_train)} / {len(X_corp_test)}")

Split Summary:
Retail Train/Test: 800 / 200
Corporate Train/Test: 73638 / 18410


## Step 4: Advanced Imputation Pipeline
For the **Corporate Track**, which exhibits structural missingness (~12%), simple mean/median fills can distort financial ratios. We use `IterativeImputer` (MICE), which models each feature as a function of others. For the **Retail Track**, which is largely complete, we use `SimpleImputer` with the median strategy to preserve robustness.

In [7]:
# Separate numeric features for imputation (only numeric columns usually have missing data in these sets)
num_cols_corp = X_corp_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols_ret = X_ret_train.select_dtypes(include=[np.number]).columns.tolist()

# 1. Corporate Imputation (MICE)
mice_imputer = IterativeImputer(random_state=42, max_iter=10)
X_corp_train[num_cols_corp] = mice_imputer.fit_transform(X_corp_train[num_cols_corp])
X_corp_test[num_cols_corp] = mice_imputer.transform(X_corp_test[num_cols_corp])

# 2. Retail Imputation (Median)
median_imputer = SimpleImputer(strategy='median')
X_ret_train[num_cols_ret] = median_imputer.fit_transform(X_ret_train[num_cols_ret])
X_ret_test[num_cols_ret] = median_imputer.transform(X_ret_test[num_cols_ret])

print("Imputation Complete.")

Imputation Complete.


## Step 5: Encoding & Robust Scaling
Credit risk data often contains extreme outliers (e.g., highly leveraged firms or high-income individuals). `StandardScaler` is sensitive to these; hence we utilize `RobustScaler`, which uses the interquartile range (IQR). Simultaneously, categorical indicators are converted to binary vectors via `OneHotEncoder`.

In [8]:
from sklearn.compose import ColumnTransformer

# Identify categorical columns
cat_cols_corp = X_corp_train.select_dtypes(include=['object']).columns.tolist()
cat_cols_ret = X_ret_train.select_dtypes(include=['object']).columns.tolist()

# Define Preprocessor for Corporate Track
corp_preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_cols_corp),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_corp)
    ])

# Define Preprocessor for Retail Track
ret_preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_cols_ret),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_ret)
    ])

# Fit and transform training data, only transform test data
X_corp_train_final = corp_preprocessor.fit_transform(X_corp_train)
X_corp_test_final = corp_preprocessor.transform(X_corp_test)

X_ret_train_final = ret_preprocessor.fit_transform(X_ret_train)
X_ret_test_final = ret_preprocessor.transform(X_ret_test)

print(f"Final Corporate Training Shape: {X_corp_train_final.shape}")
print(f"Final Retail Training Shape: {X_ret_train_final.shape}")

Final Corporate Training Shape: (73638, 23)
Final Retail Training Shape: (800, 58)


## Step 6: Targeted Class Balancing (SMOTE)
Our Corporate dataset has a severe minority class (approx. 10% default). We use **Synthetic Minority Over-sampling Technique (SMOTE)** to generate synthetic examples for the minority class. 
**CRITICAL NOTE:** Resampling is applied **ONLY** to the training data. The testing data must remain in its original, imbalanced state to provide an authentic evaluation of real-world model performance.

In [9]:
smote = SMOTE(random_state=42)

print(f"Corporate target before SMOTE:\n{y_corp_train.value_counts()}")

# Apply SMOTE to Corporate Training Data
X_corp_train_res, y_corp_train_res = smote.fit_resample(X_corp_train_final, y_corp_train)

print(f"\nCorporate target after SMOTE:\n{y_corp_train_res.value_counts()}")
print(f"Resampled Corporate Shape: {X_corp_train_res.shape}")

Corporate target before SMOTE:
Risk
0    66363
1     7275
Name: count, dtype: int64

Corporate target after SMOTE:
Risk
0    66363
1    66363
Name: count, dtype: int64
Resampled Corporate Shape: (132726, 23)


## Step 7: Modular Data Serialization
To maintain repository modularity, we serialize the preprocessed training and testing sets. This enables standalone model iteration in **Notebook 04** and **Notebook 05** without necessitating a re-run of the initial data ingestion and engineering pipeline.

In [10]:
PROCESSED_DATA_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# 1. Save Corporate Balanced Data
joblib.dump(X_corp_train_res, os.path.join(PROCESSED_DATA_DIR, 'X_corp_train_res.joblib'))
joblib.dump(y_corp_train_res, os.path.join(PROCESSED_DATA_DIR, 'y_corp_train_res.joblib'))
joblib.dump(X_corp_test_final, os.path.join(PROCESSED_DATA_DIR, 'X_corp_test_final.joblib'))
joblib.dump(y_corp_test, os.path.join(PROCESSED_DATA_DIR, 'y_corp_test.joblib'))

# 2. Save Retail Preprocessed Data
joblib.dump(X_ret_train_final, os.path.join(PROCESSED_DATA_DIR, 'X_ret_train_final.joblib'))
joblib.dump(y_ret_train, os.path.join(PROCESSED_DATA_DIR, 'y_ret_train.joblib'))
joblib.dump(X_ret_test_final, os.path.join(PROCESSED_DATA_DIR, 'X_ret_test_final.joblib'))
joblib.dump(y_ret_test, os.path.join(PROCESSED_DATA_DIR, 'y_ret_test.joblib'))

print(f"All processed datasets successfully serialized to {PROCESSED_DATA_DIR}")

All processed datasets successfully serialized to ..\data\processed


### Key Findings
*Preprocessing Pipeline & Statistical Integrity Results:*
- **Leakage-Safe Partitioning:** Confirmed 80/20 split across both tracks. Retail (800/200) and Corporate (73,638/18,410) records were partitioned prior to any feature transformation.
- **Structural Imputation (MICE):** Successfully executed the IterativeImputer on the corporate track, resolving the 12% missingness while maintaining multivariate correlations.
- **Synthetic Augmentation (SMOTE):** Successfully rectified the 90/10 corporate imbalance, augmenting the training minority class from 7,275 to 66,363 instances to achieve balanced 1:1 class parity.


---
### End of Notebook 03 — Feature Engineering and Preprocessing

**Outputs produced:**
- Preprocessed & Balanced Corporate Datasets (`X_corp_train_res.joblib`, `y_corp_train_res.joblib`, etc.)
- Preprocessed Retail Datasets (`X_ret_train_final.joblib`, `y_ret_train.joblib`, etc.)
- Advanced imputation pipelines encapsulated in memory.

**Next step → Notebook 04:** Individual Risk Modeling. Fitting high-capacity ensemble classifiers (XGBoost, Random Forest) on the processed Retail and Corporate tracks.

<div align="center"><sub>END OF NOTEBOOK 03</sub></div>